# 02 - Cohort & Retention Analysis

Weekly cohort retention, DAU/MAU stickiness, and churn -- a visual companion
to `database/queries/03_retention_analysis.sql` and
`04_churn_analysis.sql`.

**Caveat carried over from the SQL file:** absolute retention % runs high
across the board because `sessions.py` spreads a user's sessions uniformly
across their whole tenure rather than modeling decay over time. Treat the
*relative* power-user lift as the signal, not the exact percentages.

In [1]:
import os

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine

load_dotenv(find_dotenv())

engine = create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQUENTIAL_BLUE = ["#cde2fb", "#9ec5f4", "#5598e7", "#2a78d6", "#184f95"]

pd.options.display.float_format = "{:,.2f}".format

## Cohort retention heatmap

Cohorted by signup **month** here (rather than week, as in the SQL file) purely
for a legible grid size in a static chart -- same "active = had a session"
definition as `03_retention_analysis.sql` (3a).

In [2]:
cohort = pd.read_sql("""
    WITH cohorts AS (
        SELECT user_id, DATE_TRUNC('month', signup_date)::DATE AS cohort_month
        FROM users
    ),
    cohort_sizes AS (
        SELECT cohort_month, COUNT(*) AS cohort_size
        FROM cohorts
        GROUP BY cohort_month
    ),
    user_weeks AS (
        SELECT DISTINCT
            s.user_id,
            c.cohort_month,
            FLOOR(EXTRACT(EPOCH FROM (s.started_at - c.cohort_month)) / (7 * 86400))::INT AS week_number
        FROM sessions s
        JOIN cohorts c ON c.user_id = s.user_id
        WHERE s.started_at >= c.cohort_month
    )
    SELECT
        c.cohort_month,
        uw.week_number,
        ROUND(100.0 * COUNT(DISTINCT uw.user_id) / cs.cohort_size, 1) AS retention_pct
    FROM cohort_sizes cs
    JOIN cohorts c ON c.cohort_month = cs.cohort_month
    JOIN user_weeks uw ON uw.user_id = c.user_id AND uw.cohort_month = c.cohort_month
    WHERE uw.week_number BETWEEN 0 AND 8
    GROUP BY c.cohort_month, cs.cohort_size, uw.week_number
    ORDER BY c.cohort_month, uw.week_number
""", engine)

pivot = cohort.pivot(index="cohort_month", columns="week_number", values="retention_pct")

fig = px.imshow(
    pivot,
    color_continuous_scale=SEQUENTIAL_BLUE,
    labels=dict(x="Weeks since signup", y="Signup cohort", color="Retention %"),
    text_auto=True,
    aspect="auto",
    title="Monthly Cohort Retention (% active by week)",
)
fig.update_layout(template="plotly_white")
fig.show()

## DAU/MAU stickiness

In [3]:
stickiness = pd.read_sql("""
    WITH daily_actives AS (
        SELECT DATE(started_at) AS day, COUNT(DISTINCT user_id) AS dau
        FROM sessions
        GROUP BY DATE(started_at)
    ),
    avg_dau_by_month AS (
        SELECT DATE_TRUNC('month', day)::DATE AS month, AVG(dau) AS avg_dau
        FROM daily_actives
        GROUP BY DATE_TRUNC('month', day)::DATE
    ),
    monthly_actives AS (
        SELECT DATE_TRUNC('month', started_at)::DATE AS month, COUNT(DISTINCT user_id) AS mau
        FROM sessions
        GROUP BY DATE_TRUNC('month', started_at)::DATE
    )
    SELECT m.month, ROUND(100.0 * a.avg_dau / m.mau, 1) AS stickiness_pct
    FROM monthly_actives m
    JOIN avg_dau_by_month a ON a.month = m.month
    ORDER BY m.month
""", engine)

fig = px.line(
    stickiness, x="month", y="stickiness_pct", markers=True,
    color_discrete_sequence=[CATEGORICAL[0]],
    title="DAU/MAU Stickiness",
)
fig.update_layout(template="plotly_white", yaxis_title="Stickiness %", xaxis_title=None)
fig.show()

## Churn: by plan, and power users vs everyone else

Only counts a subscription as churned if it canceled *after* converting to
paid (`canceled_at > trial_end_at`) -- a trial that expired without ever
converting isn't churn, since there was nothing to churn from. Same
definition as `04_churn_analysis.sql`.

In [4]:
churn_by_plan = pd.read_sql("""
    SELECT
        plan,
        ROUND(100.0 * COUNT(*) FILTER (WHERE canceled_at > trial_end_at) / COUNT(*), 1) AS churn_rate_pct
    FROM subscriptions
    WHERE status = 'active' OR canceled_at > trial_end_at
    GROUP BY plan
""", engine)

fig = px.bar(
    churn_by_plan, x="plan", y="churn_rate_pct",
    color="plan", color_discrete_sequence=CATEGORICAL,
    title="Post-Conversion Churn Rate by Plan",
)
fig.update_layout(template="plotly_white", showlegend=False, xaxis_title=None, yaxis_title="Churn %")
fig.show()

In [5]:
power_user_churn = pd.read_sql("""
    WITH key_feature_totals AS (
        SELECT user_id, COUNT(*) AS kf_uses
        FROM events
        WHERE event_type = 'feature_used' AND feature = 'report_export'
        GROUP BY user_id
    ),
    paid_users AS (
        SELECT
            a.user_id,
            s.canceled_at,
            s.trial_end_at,
            COALESCE(k.kf_uses, 0) >= 5 AS is_power_user
        FROM subscriptions s
        JOIN accounts a ON a.account_id = s.account_id
        LEFT JOIN key_feature_totals k ON k.user_id = a.user_id
        WHERE s.status = 'active' OR s.canceled_at > s.trial_end_at
    )
    SELECT
        CASE WHEN is_power_user THEN 'power user' ELSE 'everyone else' END AS segment,
        ROUND(100.0 * COUNT(*) FILTER (WHERE canceled_at > trial_end_at) / COUNT(*), 1) AS churn_rate_pct
    FROM paid_users
    GROUP BY is_power_user
""", engine)

fig = px.bar(
    power_user_churn, x="segment", y="churn_rate_pct",
    color="segment", color_discrete_sequence=CATEGORICAL,
    title="Churn Rate: report_export Power Users vs Everyone Else",
)
fig.update_layout(template="plotly_white", showlegend=False, xaxis_title=None, yaxis_title="Churn %")
fig.show()

## Takeaways

- Retention looks high in absolute terms across every cohort -- expected given
  the flat-engagement session model (see the caveat at the top of this
  notebook), not evidence of an unusually sticky product.
- `pro` and `enterprise` churn at similar-looking post-conversion rates in
  this run; `04_churn_analysis.sql` (4b) breaks down time-to-churn for a
  finer-grained view.
- `report_export` power users churn meaningfully less than everyone else,
  consistent with the seeded `POWER_USER_RETENTION_LIFT` -- the same signal
  `05_feature_analysis.sql` finds on the conversion side.